# Actividad: Mapas Coropléticos de Migración en la Región Metropolitana

**Curso: Análisis de Datos Espaciales con Python**  
**Duración: 1 hora (máximo)**

---

## Contexto

En la clase anterior aprendimos a construir **mapas coropléticos** usando datos del PIB per cápita de México: cómo clasificar una variable continua en categorías discretas y pintar cada polígono según su clase.

Hoy vamos a **aplicar esas mismas técnicas** a un problema real y cercano: la **migración internacional en la Región Metropolitana de Chile**, usando datos del **Censo de Población y Vivienda 2024** publicados por el INE.

### Objetivos de aprendizaje

1. Practicar la construcción de mapas coropléticos con datos reales chilenos.
2. Comparar al menos 3 clasificadores (Intervalos Iguales, Cuantiles, Fisher-Jenks) sobre una misma variable.
3. Analizar críticamente qué revela (y qué oculta) cada clasificación sobre los patrones de migración.
4. Interpretar la distribución espacial de la inmigración a nivel comunal en Santiago.

### Datos

Usaremos un dataset pre-procesado a partir del Censo 2024, disponible en el repositorio:  
`https://github.com/daniopitz/censo2024`

El notebook de referencia (`02_censo_personas_inmigracion_rm_od.ipynb`) contiene el procesamiento original de los microdatos censales. Para esta actividad trabajaremos con datos **ya agregados a nivel comunal** que construiremos a partir de los diccionarios del censo.


---

## PARTE 1: Preparación del entorno y datos (15 minutos)

### 1.1 Importar librerías


In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import mapclassify
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 96


### 1.2 Datos de migración por comuna (Censo 2024)

Los siguientes datos provienen del Censo 2024 (INE) y representan el **número de personas nacidas en el extranjero** residentes en cada comuna de la Región Metropolitana. Fueron procesados originalmente en el repositorio [daniopitz/censo2024](https://github.com/daniopitz/censo2024).

Vamos a construir el DataFrame directamente para no depender de archivos Parquet pesados.


In [2]:
import pandas as pd
import dask.dataframe as dd


### 1.3 Cargar la geometría comunal

Necesitamos un mapa con los polígonos de las comunas de la RM. Usaremos la cartografía oficial disponible en formato GeoJSON/Shapefile.


In [3]:
# Cargar el shapefile de comunas de la RM

comunas_gdf = gpd.read_file("datos/external/censo2017/R13/COMUNA_C17.shp")

# Veamos qué columnas tiene

comunas_gdf.head()


,REGION,NOM_REGION,PROVINCIA,NOM_PROVIN,COMUNA,NOM_COMUNA,SHAPE_Leng,SHAPE_Area,geometry
0,13,REGIÓN METROPOLITANA DE SANTIAGO,134,MAIPO,13404,PAINE,1.625330,0.066035,"POLYGON ((-70.61889 -33.73808, -70.61811 -33.7..."
1,13,REGIÓN METROPOLITANA DE SANTIAGO,134,MAIPO,13402,BUIN,0.884164,0.021166,"POLYGON ((-70.63192 -33.64634, -70.63207 -33.6..."
2,13,REGIÓN METROPOLITANA DE SANTIAGO,131,SANTIAGO,13124,PUDAHUEL,0.720176,0.019124,"POLYGON ((-70.78914 -33.36153, -70.78824 -33.3..."
3,13,REGIÓN METROPOLITANA DE SANTIAGO,131,SANTIAGO,13103,CERRO NAVIA,0.170180,0.001076,"POLYGON ((-70.71927 -33.41334, -70.71888 -33.4..."
4,13,REGIÓN METROPOLITANA DE SANTIAGO,133,CHACABUCO,13301,COLINA,1.692007,0.093820,"POLYGON ((-70.59630 -32.95138, -70.59673 -32.9..."


In [4]:
census = dd.read_parquet('datos/external/censo2024/personas/part-0.parquet')
census.head()

,id_vivienda,id_hogar,id_persona,provincia,comuna,comuna_bajo_umbral,area,tipo_operativo,sexo,edad,...,p45_medio_transporte,p46a_tot_hijs_nac,p46b_hijas_nac,p46c_hijos_nac,p47a_tot_hijs_sobrev,p47b_hijas_sobrev,p47c_hijos_sobrev,p48_anio_nac_uh,p48_mes_nac_uh,div_genero
0,54539,1,6,131,13129,2,1,2,2,61,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
1,54542,1,1,131,13110,2,1,2,1,53,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
2,54549,1,1,134,13402,2,2,2,1,34,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,54549,1,2,134,13402,2,2,2,2,31,...,1.0,2.0,2.0,0.0,2.0,2.0,0.0,2024.0,1.0,2.0
4,54549,1,3,134,13402,2,2,2,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.4 Unir datos de migración con la geometría

Debemos hacer un **merge** (unión) entre nuestra tabla de migración y el GeoDataFrame con la geometría. Para ello necesitamos una columna en común — el nombre de la comuna.

> **Nota:** En la práctica, es mejor unir por códigos (ej: código comunal INE) en vez de por nombre, ya que los nombres pueden tener diferencias de escritura (tildes, mayúsculas, etc.).


In [5]:
migrant_reg_counts = census.groupby(['p27_nacionalidad_esp', 'comuna']).size().compute()
migrant_reg_counts


p27_nacionalidad_esp  comuna
-99                   13101     3348
                      13102      901
                      13103     1114
                      13104      611
                      13105      910
                                ... 
 862                  13601     1963
                      13602      330
                      13603      351
                      13604     2805
                      13605     1414
Length: 757, dtype: int64

In [6]:
migrant_reg_counts=migrant_reg_counts.reset_index()
migrant_reg_counts

,p27_nacionalidad_esp,comuna,0
0,-99,13101,3348
1,-99,13102,901
2,-99,13103,1114
3,-99,13104,611
4,-99,13105,910
...,...,...,...
752,862,13601,1963
753,862,13602,330
754,862,13603,351
755,862,13604,2805


In [7]:
migrant_reg_counts.rename(columns={'p27_nacionalidad_esp':'CODIGO_PAIS', 0:'Nº_MIGRANTES'},inplace=True)
migrant_reg_counts.head()

,CODIGO_PAIS,comuna,Nº_MIGRANTES
0,-99,13101,3348
1,-99,13102,901
2,-99,13103,1114
3,-99,13104,611
4,-99,13105,910


In [8]:
country_codes = {
    -99: 'No responde',
    2: 'África',
    5: 'Otros países de América del Sur',
    9: 'Oceanía',
    13: 'Otros países de América Central y El Caribe',
    21: 'América del Norte',
    32: 'Argentina',
    68: 'Bolivia (Estado Plurinacional de)',
    142: 'Asia',
    150: 'Europa',
    152: 'Chile',
    170: 'Colombia',
    332: 'Haití',
    604: 'Perú',
    862: 'Venezuela (República Bolivariana de)'
}

In [9]:

comunas = {
    13101: "Santiago",
    13102: "Cerrillos",
    13103: "Cerro Navia",
    13104: "Conchalí",
    13105: "El Bosque",
    13106: "Estación Central",
    13107: "Huechuraba",
    13108: "Independencia",
    13109: "La Cisterna",
    13110: "La Florida",
    13111: "La Granja",
    13112: "La Pintana",
    13113: "La Reina",
    13114: "Las Condes",
    13115: "Lo Barnechea",
    13116: "Lo Espejo",
    13117: "Lo Prado",
    13118: "Macul",
    13119: "Maipú",
    13120: "Ñuñoa",
    13121: "Pedro Aguirre Cerda",
    13122: "Peñalolén",
    13123: "Providencia",
    13124: "Pudahuel",
    13125: "Quilicura",
    13126: "Quinta Normal",
    13127: "Recoleta",
    13128: "Renca",
    13129: "San Joaquín",
    13130: "San Miguel",
    13131: "San Ramón",
    13132: "Vitacura",
    13201: "Puente Alto",
    13202: "Pirque",
    13203: "San José de Maipo",
    13301: "Colina",
    13302: "Lampa",
    13303: "Tiltil",
    13401: "San Bernardo",
    13402: "Buin",
    13403: "Calera de Tango",
    13404: "Paine",
    13501: "Melipilla",
    13502: "Alhué",
    13503: "Curacaví",
    13504: "María Pinto",
    13505: "San Pedro",
    13601: "Talagante",
    13602: "El Monte",
    13603: "Isla de Maipo",
    13604: "Padre Hurtado",
    13605: "Peñaflor"
}
paises={'Venezuela (República Bolivariana de)':'Venezuela', 'Bolivia (Estado Plurinacional de)':'Bolivia'}

In [10]:
migrant_reg_counts['CONTINENTE'] = migrant_reg_counts['CODIGO_PAIS'].map(country_codes)
migrant_reg_counts['COMUNA']= migrant_reg_counts['comuna'].map(comunas)
migrant_reg_counts=migrant_reg_counts.drop(columns=['comuna'])
migrant_reg_counts

,CODIGO_PAIS,Nº_MIGRANTES,CONTINENTE,COMUNA
0,-99,3348,No responde,Santiago
1,-99,901,No responde,Cerrillos
2,-99,1114,No responde,Cerro Navia
3,-99,611,No responde,Conchalí
4,-99,910,No responde,El Bosque
...,...,...,...,...
752,862,1963,Venezuela (República Bolivariana de),Talagante
753,862,330,Venezuela (República Bolivariana de),El Monte
754,862,351,Venezuela (República Bolivariana de),Isla de Maipo
755,862,2805,Venezuela (República Bolivariana de),Padre Hurtado


In [11]:
migrant_reg_counts = migrant_reg_counts[(migrant_reg_counts['CODIGO_PAIS'] != -99) & (migrant_reg_counts['CODIGO_PAIS'] != 152) ]
migrant_reg_counts

,CODIGO_PAIS,Nº_MIGRANTES,CONTINENTE,COMUNA
52,2,90,África,Santiago
53,2,4,África,Cerrillos
54,2,13,África,Cerro Navia
55,2,5,África,Conchalí
56,2,4,África,El Bosque
...,...,...,...,...
752,862,1963,Venezuela (República Bolivariana de),Talagante
753,862,330,Venezuela (República Bolivariana de),El Monte
754,862,351,Venezuela (República Bolivariana de),Isla de Maipo
755,862,2805,Venezuela (República Bolivariana de),Padre Hurtado


In [12]:
migrant_reg_counts['CONTINENTE']=migrant_reg_counts['CONTINENTE'].replace(paises)
migrant_reg_counts



/var/folders/4m/3wpqgpmj1mj_hgdh7mk_1g380000gn/T/ipykernel_1898/1872544107.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  migrant_reg_counts['CONTINENTE']=migrant_reg_counts['CONTINENTE'].replace(paises)


,CODIGO_PAIS,Nº_MIGRANTES,CONTINENTE,COMUNA
52,2,90,África,Santiago
53,2,4,África,Cerrillos
54,2,13,África,Cerro Navia
55,2,5,África,Conchalí
56,2,4,África,El Bosque
...,...,...,...,...
752,862,1963,Venezuela,Talagante
753,862,330,Venezuela,El Monte
754,862,351,Venezuela,Isla de Maipo
755,862,2805,Venezuela,Padre Hurtado


In [13]:
migrant_reg_counts.to_csv('datos/external/censo2024/personas/migrant_reg_counts.csv', index=False)